# Does either encoding move the model beyond the noise band?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import os

import polars as pl
from sklearn.metrics import average_precision_score, roc_auc_score

from fraud_detection.core.config import resolve_repo_path
from fraud_detection.core.feature_contract import FeatureContract
from fraud_detection.core.feature_contract.admission import load_admission_rules
from fraud_detection.core.schema import (
    CLIENT_ENTITY_ANCHOR,
    CLIENT_ENTITY_COMPONENTS,
    MODEL_INPUT_TABLE,
    SPLIT_TABLE,
)
from fraud_detection.evaluation.entity_purity import Anchor, EntityKey, seen_entity_flag
from fraud_detection.feature_engineering.derivations import apply_derivations, load_frequency_maps
from fraud_detection.training.data import load_raw_split, prepare_features
from fraud_detection.training.model import train_lightgbm

# Same modules the pipeline runs. That is the point of the layering rule in
# docs/code-structure.md: this notebook cannot measure a different implementation from the
# one that gets promoted, because there is only one.
PROJECT = os.environ["GCP_PROJECT_ID"]
SEEDS = [42, 7, 1337, 2024, 91]

In [2]:
rules = load_admission_rules()
declared = pl.DataFrame(
    [{"name": d.name, "tool": d.tool, "input": d.inputs[0]} for d in rules.derivations]
)
print(declared.group_by("tool").len().sort("len", descending=True))

maps = load_frequency_maps()
summary = pl.DataFrame(
    [
        {"column": c, "levels_in_map": len(t), "one_hot_columns_this_would_need": len(t)}
        for c, t in maps.items()
    ]
).sort("levels_in_map", descending=True)
summary

shape: (3, 2)
┌─────────────────────────┬─────┐
│ tool                    ┆ len │
│ ---                     ┆ --- │
│ str                     ┆ u32 │
╞═════════════════════════╪═════╡
│ one_hot                 ┆ 18  │
│ days_since_to_start_day ┆ 7   │
│ frequency_encode        ┆ 5   │
└─────────────────────────┴─────┘


column,levels_in_map,one_hot_columns_this_would_need
str,i64,i64
"""DeviceInfo""",1176,1176
"""addr1""",202,202
"""id_31""",99,99
"""R_emaildomain""",60,60
"""P_emaildomain""",59,59


Read the last column as the counterfactual: one-hot on `DeviceInfo` alone would add over a
thousand columns, each of them 1 on a handful of rows. Frequency encoding adds **one**, and
it is an integer a tree can put a threshold on. That asymmetry is why the two families are
declared over disjoint column sets rather than compared head to head on the same columns.


Three variants over one load: the contract as promoted, plus and minus each encoding
family. Loading once and projecting three ways is what keeps this honest — the splits, the
cold-entity flag and the row order are identical across variants, so the only thing that
differs is the column set.

In [3]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT)
contract = FeatureContract.from_json(resolve_repo_path("references/feature-contract.json").read_text())
tables = {"model_input_table": MODEL_INPUT_TABLE, "split_table": SPLIT_TABLE}
raw = {s: load_raw_split(client, PROJECT, s, **tables) for s in ("train", "val", "test")}

key = EntityKey(columns=CLIENT_ENTITY_COMPONENTS, anchors=(Anchor(CLIENT_ENTITY_ANCHOR),))
seen = {s: seen_entity_flag(raw["train"], f, key).fill_null(False).cast(pl.Boolean)
        for s, f in raw.items()}

derived = {s: apply_derivations(f, rules.derivations) for s, f in raw.items()}
print({s: (raw[s].width, derived[s].width) for s in raw})

{'train': (476, 506), 'val': (476, 506), 'test': (476, 506)}


In [4]:
ONE_HOT = [d.name for d in rules.derivations if d.tool == "one_hot"]
FREQ = [d.name for d in rules.derivations if d.tool == "frequency_encode"]
admitted = [c for c in contract.training_features() if c in derived["train"].columns]

VARIANTS = {
    "baseline": [c for c in admitted if c not in ONE_HOT and c not in FREQ],
    "+ one_hot": [c for c in admitted if c not in FREQ],
    "+ frequency": [c for c in admitted if c not in ONE_HOT],
    "+ both": admitted,
}
{name: len(cols) for name, cols in VARIANTS.items()}

{'baseline': 183, '+ one_hot': 201, '+ frequency': 187, '+ both': 205}

> A variant's column list is drawn from **what the contract admitted**, not from what was
> declared. If an audit rejected an encoded column, it is absent here too — the question
> being asked is "does this encoding help the model the pipeline would actually promote",
> not "does it help if we overrule the audits". Which columns survived is in §4.


The expensive cell. 4 variants × 5 seeds, hyperparameters pinned to the configuration
the production run settled on so the only moving parts are the column set and the seed.

In [ ]:
WINNING = {"num_leaves": [96], "learning_rate": [0.05], "feature_fraction": [0.6],
           "bagging_fraction": [0.7], "min_child_samples": [80]}


def split_on(columns, name):
    from fraud_detection.training.data import SplitFrame

    frame = derived[name]
    features = prepare_features(frame).select(columns)
    return SplitFrame(
        features=features,
        labels=frame.get_column("isFraud").cast(pl.Int8),
        amounts=frame.get_column("TransactionAmt").cast(pl.Float64),
        seen_in_train=seen[name],
    )


rows = []
for variant, columns in VARIANTS.items():
    splits = {s: split_on(columns, s) for s in ("train", "val", "test")}
    y = splits["test"].labels.to_numpy()
    for seed in SEEDS:
        m = train_lightgbm(splits["train"], splits["val"], splits["test"],
                           search_space=WINNING, n_iter=1, seed=seed)
        rows.append({
            "variant": variant, "seed": seed, "features": len(columns),
            # Raw scores. `test_roc_auc` in metrics.json is computed on the calibrated
            # probabilities, and the submission carries the raw ones.
            "roc_auc": roc_auc_score(y, m.test_scores),
            "pr_auc": average_precision_score(y, m.test_scores),
            "best_iteration": m.booster.best_iteration,
        })
        print(rows[-1], flush=True)

results = pl.DataFrame(rows)
results.write_parquet("encodings_results.parquet")

In [ ]:
# The two references the plot below needs: the baseline row, and the resolution of a
# five-seed mean. Both were computed in the shared analysis this notebook was split from.
NOISE_SD = {"roc_auc": 0.0029, "pr_auc": 0.0065}  # measured, five seeds
bar = {m: sd * (2 / len(SEEDS)) ** 0.5 for m, sd in NOISE_SD.items()}

agg = results.group_by("variant").agg(
    pl.col("features").first(),
    pl.col("roc_auc").mean().alias("roc_mean"),
    pl.col("roc_auc").std().alias("roc_sd"),
    pl.col("pr_auc").mean().alias("pr_mean"),
    pl.col("pr_auc").std().alias("pr_sd"),
)
base = agg.filter(pl.col("variant") == "baseline").row(0, named=True)
print(f"resolution of a 5-seed mean: ROC-AUC +-{bar['roc_auc']:.4f}")
agg


In [ ]:
import plotly.express as px

# 1. Plot ROC-AUC points across variants (5 seeds each)
fig_roc = px.strip(
    results.to_pandas(),
    x="variant", 
    y="roc_auc", 
    color="variant", 
    title="ROC-AUC across 5 Seeds by Variant",
    stripmode="overlay"
)
fig_roc.update_traces(marker=dict(size=8, opacity=0.8))

# Add a horizontal region for the noise band around the baseline mean
baseline_roc = base["roc_mean"]
roc_bar = bar["roc_auc"]
fig_roc.add_hrect(
    y0=baseline_roc - roc_bar, 
    y1=baseline_roc + roc_bar, 
    line_width=0, fillcolor="gray", opacity=0.2,
    annotation_text="Baseline Noise Band", annotation_position="top right"
)
fig_roc.show()

# 2. Plot PR-AUC points across variants (5 seeds each)
fig_pr = px.strip(
    results.to_pandas(),
    x="variant", 
    y="pr_auc", 
    color="variant", 
    title="PR-AUC across 5 Seeds by Variant",
    stripmode="overlay"
)
fig_pr.update_traces(marker=dict(size=8, opacity=0.8))

# Add a horizontal region for the noise band around the baseline mean
baseline_pr = base["pr_mean"]
pr_bar = bar["pr_auc"]
fig_pr.add_hrect(
    y0=baseline_pr - pr_bar, 
    y1=baseline_pr + pr_bar, 
    line_width=0, fillcolor="gray", opacity=0.2,
    annotation_text="Baseline Noise Band", annotation_position="top right"
)
fig_pr.show()
